In [1]:
from sklearn import svm

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler as  StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, balanced_accuracy_score
from sklearn.model_selection import KFold, cross_validate, cross_val_score, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from imblearn.pipeline import Pipeline as im_Pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import roc_curve, confusion_matrix

In [2]:
df1 = pd.read_csv("master_tripel_cramps_dataset.csv")

df_clean_og = df1.dropna(subset=[
    "cramps",
    "cramps binary",
    "cramps group",
    "phase",
    "lh",
    "estrogen",
    "nightly_temperature",
    "hr_mean",
    "glucose_median",
    "glucose_std"
]).copy()

cramps_map_menstrual = {
    "Not at all": 0,
    "Very Low/Little": 1,
    "Low": 2,
    "Moderate": 3,
    "High": 4,
    "Very High": 5
}

cramps_map_binary_menstrual = {
    "Low Pain": 0,
    "High Pain": 1,
}

cramps_map_group_menstrual = {
    "No Pain": 0,
    "Low Pain": 1,
    "High Pain": 2,
}

df_clean_og["cramps"] = df_clean_og["cramps"].map(cramps_map_menstrual)
df_clean_og["cramps binary"] = df_clean_og["cramps binary"].map(cramps_map_binary_menstrual)
df_clean_og["cramps group"] = df_clean_og["cramps group"].map(cramps_map_group_menstrual)

df_clean_og = df_clean_og.dropna(subset=["cramps binary"]).copy()
drop_cols = [
    "id",
    "cramps",
    "cramps binary",
    "cramps group",
    "study_interval_x",
    "study_interval_y",
    "day_in_study"
]

X = df_clean_og.drop(columns=drop_cols)
y = df_clean_og["cramps group"].astype(int)

variables_categoricas = ["phase"]
variables_num = [
    col for col in X.columns
    if col not in variables_categoricas]

In [3]:
groups = df_clean_og["id"]

cv_outer = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_inner = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

import pickle 
with open("outer_splits_binary.pkl", "rb") as f:
    outer_splits = pickle.load(f)

def get_best_rho(y_train, y_train_proba_oof, rhos=np.linspace(0.1, 0.9, 17)):

    best_rho = None
    best_f1 = -1

    for rho in rhos:

        y_pred = (y_train_proba_oof >= rho).astype(int)

        f1 = f1_score(y_train, y_pred, average="macro")

        if f1 > best_f1:
            best_f1 = f1
            best_rho = rho

    return best_rho

In [4]:
param_grid = {"KNN": {
                    "classifier__n_neighbors": [3, 5, 7, 9, 11],
                    "classifier__weights": ["uniform", "distance"],
                    "classifier__metric": ["euclidean", "manhattan"],
                    "feature_selection__k": [5, 10, 15, 20, "all"]}, 
              "LogReg": {
                    "classifier__C": [0.001, 0.01, 0.1, 1, 10],
                    "classifier__class_weight": ["balanced"],
                    "classifier__penalty": ["l1", "l2"],
                    "feature_selection__k": [5, 10, 15, 20, "all"]},
              "Random_Forest": {
                    "classifier__n_estimators": [10, 20, 50, 100, 200, 250, 300, 500],
                    "classifier__max_depth": [1, 2, 3, 5, 7, 10],
                    "classifier__min_samples_leaf": [1, 5, 10],
                    "classifier__max_features": ["sqrt", "log2"],
                    "classifier__max_leaf_nodes": [2, 4, 8, 16, 32],
                    "classifier__class_weight": ["balanced"]},
              "SVM":{
                    "classifier__kernel": ["linear","rbf"],
                    "classifier__gamma": ["scale", "auto"],
                    "classifier__C": [0.01, 0.1, 1, 10],
                    "classifier__class_weight": ["balanced"],
                    "feature_selection__k": [5, 10, 15, 20, "all"]}

    }

In [5]:
model_df = []
type_response_df = []
accuracy_model = []
balanced_accuracy_model = []
f1_macro_model = []
f1_weighted_model = []
models = ["KNN", "LogReg","Random_Forest","SVM"]


accuracy = np.nan * np.ones((len(models), len(outer_splits)))
balanced_accuracy = np.nan * np.ones((len(models), len(outer_splits)))
f1_macro = np.nan * np.ones((len(models), len(outer_splits)))
f1_weighted = np.nan * np.ones((len(models), len(outer_splits)))

best_params_all = {model: [] for model in models}
selected_features_all = {model: [] for model in models}

for j, (train_idx, test_idx) in enumerate(outer_splits):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    groups_train = groups.iloc[train_idx]
        
    for (i,model) in enumerate(models):
        preprocessor = ColumnTransformer(
            transformers=[
                ("zscore", StandardScaler(), variables_num),
                ("ohe", OneHotEncoder(handle_unknown="ignore"), variables_categoricas)
        ])
        match model:
            case "LogReg":
                mdl = LogisticRegression( solver = "saga", max_iter=1000, random_state = 42)   
            case "Random_Forest":
                mdl = RandomForestClassifier(random_state = 42)
            case "KNN":
                mdl = KNeighborsClassifier()
            case "SVM":
                mdl = svm.SVC(probability = True, random_state = 42)
        if model == "Random_Forest":
            pipeline = Pipeline([
                ("preprocessor", preprocessor),
                ("classifier", mdl)])
        else:
            pipeline = Pipeline([
                ("preprocessor", preprocessor),
                ("feature_selection", SelectKBest(score_func=f_classif)),
                ("classifier", mdl)])

        grid = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid[model],
                cv=cv_inner,
                scoring="f1_macro",
                n_jobs=-1)
        
        grid.fit(X_train, y_train, groups=groups_train)
        
        mdl_selected = grid.best_estimator_
        best_params_all[model].append(grid.best_params_)
        
        if model != "Random_Forest":
            feature_names = mdl_selected.named_steps["preprocessor"].get_feature_names_out()
            selector = mdl_selected.named_steps["feature_selection"]
            selected_features = feature_names[selector.get_support()].tolist()
            print(selected_features)
        else:
            selected_features = X_train.columns.tolist()

        selected_features_all[model].append(selected_features)

        mdl_selected.fit(X_train, y_train)
        y_pred = mdl_selected.predict(X_test)
        y_prob = mdl_selected.predict_proba(X_test)[:, 1]
        
        accuracy[i, j] = accuracy_score(y_test, y_pred)
        balanced_accuracy[i, j] = balanced_accuracy_score(y_test, y_pred)
        f1_macro[i, j] = f1_score(y_test, y_pred, average="macro")
        f1_weighted[i, j] = f1_score(y_test, y_pred, average="weighted")

        cm = confusion_matrix(y_test, y_pred)

        print(f"\n===== Fold {j+1} - {model} - {grid.best_params_} =====")
        print(f"Accuracy: {accuracy[i,j]:.4f}")
        print(f"Balanced Accuracy: {balanced_accuracy[i,j]:.4f}")
        print(f"F1 Macro: {f1_macro[i,j]:.4f}")

        print("\nConfusion matrix:")
        print(cm)

        print("\nClassification report:")
        print(classification_report(y_test, y_pred))
        
for (i,model) in enumerate(models):
    accuracy_model.append(f"{np.mean(accuracy[i,:]):.2f} +- {np.std(accuracy[i,:]):.2f} [{np.min(accuracy[i,:]):.2f} -  {np.max(accuracy[i,:]):.2f}]")
    balanced_accuracy_model.append(f"{np.mean(balanced_accuracy[i,:]):.2f} +- {np.std(balanced_accuracy[i,:]):.2f} [{np.min(balanced_accuracy[i,:]):.2f} -  {np.max(balanced_accuracy[i,:]):.2f}]")
    f1_macro_model.append(f"{np.mean(f1_macro[i,:]):.2f} +- {np.std(f1_macro[i,:]):.2f}  [{np.min(f1_macro[i,:]):.2f}  -   {np.max(f1_macro[i,:]):.2f}]")
    f1_weighted_model.append(f"{np.mean(f1_weighted[i,:]):.2f} +- {np.std(f1_weighted[i,:]):.2f}  [{np.min(f1_weighted[i,:]):.2f}  -   {np.max(f1_weighted[i,:]):.2f}]")
        
    if i == 0:
        type_response_df.append("cramps")
    else:
        type_response_df.append("")
        
    model_df.append(model)


df_performance_models = pd.DataFrame(np.transpose([type_response_df,model_df,accuracy_model,balanced_accuracy_model,f1_macro_model,f1_weighted_model, AUC_model]), columns=["Type response","Model", "Accuracy","Balanced Accuracy","F1-Score (macro)","F1-Score (weighted)","AUC"])
display(df_performance_models)

['zscore__hr_mean', 'zscore__hr_min', 'zscore__hr_q_05', 'zscore__hr_q_25', 'ohe__phase_Menstrual']

===== Fold 1 - KNN - {'classifier__metric': 'euclidean', 'classifier__n_neighbors': 5, 'classifier__weights': 'distance', 'feature_selection__k': 5} =====
Accuracy: 0.4607
Balanced Accuracy: 0.3986
F1 Macro: 0.3812

Confusion matrix:
[[178 148  11]
 [ 76  63  35]
 [ 11  14  11]]

Classification report:
              precision    recall  f1-score   support

           0       0.67      0.53      0.59       337
           1       0.28      0.36      0.32       174
           2       0.19      0.31      0.24        36

    accuracy                           0.46       547
   macro avg       0.38      0.40      0.38       547
weighted avg       0.52      0.46      0.48       547

['zscore__estrogen', 'zscore__nightly_temperature', 'zscore__hr_mean', 'zscore__hr_median', 'zscore__hr_std', 'zscore__hr_cv', 'zscore__hr_min', 'zscore__hr_q_05', 'zscore__hr_q_25', 'zscore__hr_q_75', 'zscore__glu

NameError: name 'AUC_model' is not defined

In [6]:
df_performance_models = pd.DataFrame(np.transpose([type_response_df,model_df,accuracy_model,balanced_accuracy_model,f1_macro_model,f1_weighted_model]), columns=["Type response","Model", "Accuracy","Balanced Accuracy","F1-Score (macro)","F1-Score (weighted)"])
display(df_performance_models)

,Type response,Model,Accuracy,Balanced Accuracy,F1-Score (macro),F1-Score (weighted)
0,cramps,KNN,0.41 +- 0.05 [0.32 - 0.46],0.37 +- 0.03 [0.32 - 0.41],0.35 +- 0.03 [0.30 - 0.38],0.40 +- 0.06 [0.32 - 0.48]
1,,LogReg,0.42 +- 0.05 [0.35 - 0.48],0.45 +- 0.04 [0.40 - 0.51],0.39 +- 0.02 [0.35 - 0.41],0.41 +- 0.07 [0.33 - 0.51]
2,,Random_Forest,0.42 +- 0.08 [0.27 - 0.48],0.42 +- 0.06 [0.34 - 0.49],0.37 +- 0.05 [0.27 - 0.43],0.42 +- 0.08 [0.26 - 0.49]
3,,SVM,0.42 +- 0.06 [0.34 - 0.48],0.45 +- 0.06 [0.35 - 0.55],0.39 +- 0.04 [0.33 - 0.45],0.41 +- 0.06 [0.33 - 0.47]
